# Notebook 2 - Preprocessing

## Week 3 Summary

- Filtered the dataset by date range and property type.
- Converted features to appropriate data types.
- Handled missing values and removed outliers.
- Split the dataset into training and testing sets using the latest month as the test set.
- Applied One-Hot Encoding to selected categorical features after the train/test split to prevent data leakage.
- Exported cleaned and encoded datasets for model development.

## Setup & Load Data

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# Load csv files into dataframe
folder_path = Path("../data/raw")
csv_files = sorted(folder_path.glob("*.csv"))
raw_df_lst = []
for file in csv_files:
    temp_df = pd.read_csv(file, low_memory = False)
    raw_df_lst.append(temp_df)
raw_df = pd.concat(raw_df_lst, ignore_index = True)

## Cleaning Data

### 1. Data Filtering

#### Filtering Strategy

The dataset was restricted to records between January 2022 and May 2026. 

Only residential single-family properties were kept because the project goal is to predict close prices for single residential properties. 

This filtering step helps reduce noise from other property types such as condos, commercial properties, land, and leases.

In [3]:
# Select the start date and end date
start_month = "2022-01"
end_month = "2026-05"

In [4]:
# Filter the length of data
raw_df["CloseDate"] = pd.to_datetime(raw_df["CloseDate"], errors = "coerce")
raw_df["CloseMonth"] = raw_df["CloseDate"].dt.to_period("M")
raw_df = raw_df[(raw_df["CloseMonth"] >= start_month) & (raw_df["CloseMonth"] <= end_month)].copy()
# Filter dataframe to PropertyType = Residential and PropertySubType = Single Family Residence
raw_df_filtered = raw_df[(raw_df.PropertyType == "Residential") & (raw_df.PropertySubType == "SingleFamilyResidence")]

### 2. Datatype Cleaning

### Cleaning Strategy

After filtering the dataset to residential single-family properties, a copy of the filtered DataFrame was created for preprocessing.

The cleaning process focused on standardizing data types and removing unusable columns before handling missing values and outliers.

The following steps were applied:

- Fully empty columns were removed because they do not provide useful information for modeling.
- Date-related columns were converted to datetime format to support time-based filtering and future feature engineering.
- Boolean columns were standardized using a mapping dictionary so values such as `"Yes"`, `"No"`, `"Y"`, `"N"`, `"True"`, and `"False"` are consistently represented as boolean values.
- Latitude and longitude fields were converted to numeric values for possible geographic analysis.
- Identifier columns such as listing IDs and postal codes were converted to string type because they represent labels rather than numerical quantities.
- Integer-like property features, including bedrooms, bathrooms, year built, stories, garage spaces, and parking spaces, were converted to numeric integer format.

This step ensures that each feature has an appropriate data type before missing value handling, outlier removal, train/test splitting, and feature encoding.

In [5]:
# Copy dataframe
df = raw_df_filtered.copy()
# Drop fully empty columns
empty_cols = df.columns[df.isna().all()].tolist()
df = df.drop(columns = empty_cols)
# Clean the filtered dataframe
# Date columns (date)
date_cols = ["CloseDate", "ContractStatusChangeDate", "PurchaseContractDate", "ListingContractDate"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors = "coerce")
# Boolean columns (boolean)
bool_cols = ["ViewYN", "WaterfrontYN", "BasementYN", "PoolPrivateYN", "AttachedGarageYN", \
             "FireplaceYN", "NewConstructionYN"]
bool_map = {"True": True, "False": False, "true": True, "false": False, "Y": True, "N": False, \
            "Yes": True, "No": False, "1": True, "0": False, True: True, False: False}
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map(bool_map).astype("boolean")
# Longitude & Latitude Columns (float)
for col in ["latfilled", "lonfilled"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors = "coerce")
# ID Columns (string)
id_cols = ["ListingKey", "ListingKeyNumeric", "ListingId", "StreetNumberNumeric", "PostalCode"]
for col in id_cols:
    if col in df.columns:
        df[col] = df[col].astype("string")
# Integer Columns (int)
int_cols = ["DaysOnMarket", "YearBuilt", "BathroomsTotalInteger", "BedroomsTotal", "Stories", "MainLevelBedrooms", "GarageSpaces", "ParkingTotal"]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors = "coerce").round().astype("Int64")
# Show the information of cleaned data
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 399157 entries, 3 to 794267
Data columns (total 75 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   Flooring                     256352 non-null  object        
 1   ViewYN                       359640 non-null  boolean       
 2   WaterfrontYN                 175 non-null     boolean       
 3   BasementYN                   9747 non-null    boolean       
 4   PoolPrivateYN                358610 non-null  boolean       
 5   OriginalListPrice            398378 non-null  float64       
 6   ListingKey                   399157 non-null  string        
 7   ListAgentEmail               398296 non-null  object        
 8   CloseDate                    399157 non-null  datetime64[ns]
 9   ClosePrice                   399155 non-null  float64       
 10  ListAgentFirstName           395752 non-null  object        
 11  ListAgentLastName            39

### 3. Missing Value

#### Missing Value Strategy

Missing values were handled based on the missing rate and feature meaning:

- Columns with very high missing rates and limited modeling relevance were removed.
- Boolean features were filled with `False` when missing values likely indicated absence of the feature.
- Numeric features were filled with the median.
- Categorical features were filled with `"Unknown"`.
- Rows with very small amounts of missingness were dropped.

In [6]:
missing_summary_1 = pd.DataFrame({"missing_count": df.isna().sum(),"missing_rate": df.isna().mean()})\
                    .sort_values("missing_rate", ascending = False)
missing_summary_1[missing_summary_1.missing_rate > 0]

,missing_count,missing_rate
WaterfrontYN,398982,0.999562
BelowGradeFinishedArea,396829,0.994168
BasementYN,389410,0.975581
BuilderName,380399,0.953006
LotSizeDimensions,374649,0.938601
...,...,...
BathroomsTotalInteger,75,0.000188
ListAgentLastName,37,0.000093
PostalCode,2,0.000005
ClosePrice,2,0.000005


In [7]:
# Drop the columns with missing rate > 50% and seem to be irrelevant
drop_cols_50 = ["BelowGradeFinishedArea", "BuilderName", "LotSizeDimensions", "BuildingAreaTotal",
    "CoBuyerAgentFirstName", "ElementarySchool", "MiddleOrJuniorSchool", "HighSchool",
    "CoListAgentFirstName", "CoListAgentLastName", "CoListOfficeName", "BuyerAgencyCompensation",
    "BuyerAgencyCompensationType", "SubdivisionName", "latfilled", "lonfilled"]
df = df.drop(columns = [c for c in drop_cols_50 if c in df.columns])

In [8]:
# Fill missing values for missing rate > 50%
for col in ["WaterfrontYN", "BasementYN"]:
    df[col] = df[col].fillna(False)
df["AssociationFeeFrequency"] = df["AssociationFeeFrequency"].fillna("None")

In [9]:
# Drop the columns with agent's information or MLS's information
irrelevant_info = ["BuyerAgentAOR","ListAgentAOR", "BuyerOfficeAOR", "BuyerOfficeName", "ListAgentEmail", \
                   "BuyerAgentMlsId", "ListAgentFirstName", "BuyerAgentFirstName", "BuyerAgentLastName", \
                   "ListAgentLastName", "ListAgentFullName"]
df = df.drop(columns = [c for c in irrelevant_info if c in df.columns])

In [10]:
# Drop the missing values of the columns with missing rate < 1%
# Recalculate missing summary after dropping/filling columns
missing_summary_2 = pd.DataFrame({"missing_count": df.isna().sum(), "missing_rate": df.isna().mean()})\
                        .sort_values("missing_rate", ascending = False)
col_less_one_percent = missing_summary_2[(missing_summary_2.missing_rate < 0.01) & (missing_summary_2.missing_rate > 0)].index.tolist()
col_less_one_percent = [col for col in col_less_one_percent if col in df.columns]
df = df.dropna(subset = col_less_one_percent)

In [11]:
missing_summary_3 = pd.DataFrame({"missing_count": df.isna().sum(), "missing_rate": df.isna().mean()})\
                    .sort_values("missing_rate", ascending=False)
remain_cols = missing_summary_3[missing_summary_3.missing_rate > 0].index.tolist()
for rc in remain_cols:
    if rc not in df.columns:
        continue
    dtype = str(df[rc].dtype)
    if dtype == "boolean" or "YN" in rc:
        df[rc] = df[rc].fillna(False)
    elif dtype in ["float64", "int64", "Int64"]:
        df[rc] = df[rc].fillna(df[rc].median())
    elif "datetime" in dtype:
        continue
    elif dtype in ["object", "string"]:
        df[rc] = df[rc].fillna("Unknown")

In [12]:
missing_summary_new = pd.DataFrame({"missing_count": df.isna().sum(), "missing_rate": df.isna().mean()})\
                    .sort_values("missing_rate", ascending = False)
missing_summary_new

,missing_count,missing_rate
ContractStatusChangeDate,7882,0.019982
Flooring,0,0.000000
ViewYN,0,0.000000
YearBuilt,0,0.000000
StreetNumberNumeric,0,0.000000
ListingId,0,0.000000
BathroomsTotalInteger,0,0.000000
City,0,0.000000
BedroomsTotal,0,0.000000
PurchaseContractDate,0,0.000000


### 4. Outliers

#### Outliers Strategy

Several numerical features contained unrealistic or extreme values. 

Invalid records such as non-positive `ClosePrice` and `LivingArea` were removed. 

For key numerical features, the 1st and 99th percentiles were used to reduce the effect of extreme outliers while keeping most of the dataset.

In [13]:
# Remove basic invalid values
df = df[(df["ClosePrice"] > 0) & (df["LivingArea"] > 0) & (df["YearBuilt"].between(1800, 2026))].copy()

# Remove extreme value (less than 1st and 99th percentiles)
for col in ["ClosePrice", "LivingArea", "LotSizeSquareFeet"]:
    before = len(df)
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df = df[(df[col] >= lower) & (df[col] <= upper)].copy()
    print(f"{col}: {before} -> {len(df)}")

ClosePrice: 394318 -> 386545
LivingArea: 386545 -> 378822
LotSizeSquareFeet: 378822 -> 371247


### 5. Train/Test set split

#### Split Strategy

The most recent month of available data was used as the test set, while earlier months were used for training. 

This time-based split is more realistic than a random split because the model is trained on historical records and evaluated on future transactions.

In [14]:
# The most recent month of available data as the test set
test_df = df[df["CloseMonth"] == end_month].copy()
# The remaining data as the train set
train_df = df[df["CloseMonth"] < end_month].copy()

In [15]:
print("Cleaned data:", df.shape)
print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Train months:", train_df["CloseMonth"].min(), "to", train_df["CloseMonth"].max())
print("Test month:", str(test_df["CloseMonth"].iloc[0]))

Cleaned data: (371247, 48)
Train: (360009, 48)
Test: (11238, 48)
Train months: 2022-01 to 2026-04
Test month: 2026-05


### 6. Feature Exclusion

#### Feature Exclusion Strategy

To prevent data leakage, the following variables are excluded from the machine learning features:

- `ListPrice`
- `OriginalListPrice`

These variables are highly correlated with the target variable (`ClosePrice`) because they represent pricing information available during the transaction process. 

Including these features would artificially inflate model performance and would not reflect a realistic prediction scenario. 

This decision follows the project guideline provided by the internship supervisor.

In [16]:
target = "ClosePrice"
# Separate the target values
x_train = train_df.drop(columns = [target])
x_test = test_df.drop(columns = [target])
y_train = train_df[target]
y_test = test_df[target]
# Drop the columns that will not fit with models
# City and PostalCode are temporarily excluded due to high cardinality.
# They may be reintroduced later using target encoding, frequency encoding, or geographic grouping.
# Features excluded from modeling
drop_model_cols = [
    # Target leakage
    "ListPrice",
    "OriginalListPrice",
    # Date fields
    "CloseDate",
    "CloseMonth",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate",
    # Unique identifiers
    "ListingKey",
    "ListingKeyNumeric",
    "ListingId",
    # Address / location identifiers
    "UnparsedAddress",
    "StreetNumberNumeric",
    "PostalCode",
    # Property labels
    "PropertyType",
    "PropertySubType",
    "MlsStatus",
    # High-cardinality text
    "ListOfficeName",
    "City"
]
x_train = x_train.drop(columns = [c for c in drop_model_cols if c in x_train.columns])
x_test = x_test.drop(columns = [c for c in drop_model_cols if c in x_test.columns])

### 6. One-hot encoding

### Encoding Strategy

Categorical variables were encoded after the train/test split to avoid data leakage. 

The training and testing encoded datasets were aligned to ensure both sets have the same feature columns. 

Missing dummy columns in the test set were filled with zero.

In [17]:
cat_cols = ["CountyOrParish", "Levels", "AssociationFeeFrequency"]
cat_cols = [c for c in cat_cols if c in x_train.columns]
# Cannot apply one-hot to whole dataframe since it might cause dataleakage
x_train_encoded = pd.get_dummies(x_train, columns = cat_cols, dummy_na = False, dtype = "int8")
x_test_encoded = pd.get_dummies(x_test, columns = cat_cols, dummy_na = False, dtype = "int8")
x_train_encoded, x_test_encoded = x_train_encoded.align(x_test_encoded, join = "left", axis = 1, fill_value = 0)

In [18]:
# Training set
train_encoded = x_train_encoded.copy()
train_encoded[target] = y_train.values
# Test set
test_encoded = x_test_encoded.copy()
test_encoded[target] = y_test.values

In [19]:
print("Train encoded:", train_encoded.shape)
print("Test encoded:", test_encoded.shape)
print("Same columns:", list(train_encoded.columns) == list(test_encoded.columns))

Train encoded: (360009, 113)
Test encoded: (11238, 113)
Same columns: True


### 7. Clean CSV export

In [20]:
# Set up path for full datasets
full_dir = Path("../data/cleaned_full")
full_dir.mkdir(parents = True, exist_ok = True)
# Cleaned whole dataset
df.to_csv(full_dir / "cleaned_crmls.csv", index = False)
# Training/test datasets
train_df.to_csv(full_dir / "train_clean.csv", index = False)
test_df.to_csv(full_dir / "test_clean.csv", index = False)
# Encoded datasets
train_encoded.to_csv(full_dir / "train_encoded.csv", index = False)
test_encoded.to_csv(full_dir / "test_encoded.csv", index = False)

print("Files exported successfully.")

Files exported successfully.


In [21]:
# Set up path for sample datasets
sample_dir = Path("../data/cleaned")
sample_dir.mkdir(parents = True, exist_ok = True)
# Set up
sample_size = 10000
random_state = 42
# Sample cleaned dataset
df.sample(sample_size, random_state = random_state).to_csv(sample_dir / "cleaned_crmls.csv", index = False)
# Training/test sample datasets
train_df.sample(sample_size, random_state = random_state).to_csv(sample_dir / "train_clean.csv", index = False)
test_df.to_csv(sample_dir / "test_clean.csv", index = False)
# Encoded sample datasets
train_encoded.sample(sample_size, random_state = random_state).to_csv(sample_dir / "train_encoded.csv", index = False)
test_encoded.to_csv(sample_dir / "test_encoded.csv", index = False)

print("Files exported successfully.")

Files exported successfully.


### Exported Datasets

The preprocessing notebook exports both full and sample versions of the processed datasets. 

Full datasets are saved locally in `data/cleaned_full/` and will be used for model training and evaluation.  

Sample datasets are saved in `data/cleaned/` for GitHub demonstration purposes.

## Preprocessing Summary

After preprocessing, the dataset was cleaned, encoded, and split into training and testing sets. 

Several variables were intentionally excluded from modeling, including `ListPrice` and `OriginalListPrice`, to prevent target leakage and ensure that model performance reflects a realistic prediction setting. 

The resulting datasets are ready for baseline model development in the next notebook.

---------------------------------------------------------------------------------------------------------------------------------------------------------

## Author

Jasper Fan-Chiang

M.S. in Applied Data Science
University of Southern California

IDX Exchange — Data Science Internship